# 6.1 Networks with metadata

This notebook accompanies Section 6.1 of [*Community Detection with the Map Equation and Infomap: Theory and Applications*](https://doi.org/10.1145/3779648).

Nodes in real networks often carry extra information: a person's age, an animal's species, a category label. Section 6.1 calls this *metadata* and gives two ways to fold it into the map equation. Here you take the second route, a metadata-dependent encoding of random walks. You build a metadata-dependent encoding graph from a small network, then run Infomap on that graph to find modules that respect both the link structure and the node labels.


In [ ]:
!git clone https://github.com/mapequation/color-map-equation.git

In [ ]:
!cargo build --release --manifest-path color-map-equation/simulate/Cargo.toml

In [ ]:
import networkx as nx
from infomap import Infomap


In [ ]:
import _support as util


## A toy network

Build a seven-node network and assign each node a categorical metadata label $\ell_u$ from $\mathcal{L} = \{1, 2\}$. In the plots that follow, the label sets the node shape (square or circle) and the detected module sets the color.


In [ ]:
# The network has 7 nodes.
G = nx.Graph([(1, 2), (2, 3), (3, 1), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)])

pos = nx.spring_layout(G, seed=27)

In [ ]:
# We assign a categorical metadata label to each node
metadata = {1: 1, 2: 1, 3: 2, 4: 1, 5: 2, 6: 2, 7: 2}

nx.set_node_attributes(G, metadata, "metadata")

In [ ]:
modules = {n: 1 for n in G.nodes}
nx.set_node_attributes(G, modules, "node_color_attr")

# Visualization: metadata 1 = circles, metadata 2 = squares
util.plot_modular_network(
    G,
    pos=pos,
    node_color_attr="node_color_attr",
    figsize=(6, 4),
    edge_width=1.5,
    use_value_as_palette_index=True,
    shape_attr="metadata",
    shape_map={1: "s", 2: "o"},
)

## Modules from the standard map equation

First run Infomap on the network alone, ignoring the labels. This is the baseline: modules driven by link structure only. Compare the shapes against the colors to see where the labels cut across the structural modules.


In [ ]:
# ---------------------------------------------------------------------
# 1. Detect communities with Infomap (without metadata)
# ---------------------------------------------------------------------

# Standard Infomap run
im = Infomap(
    directed=False,  # Treat G as an undirected graph
    silent=True,  # Suppress terminal output
    num_trials=1,  # Single stochastic trial
    seed=123,  # Fix the random seed for reproducibility
)

# Load the network into Infomap directly from the NetworkX graph
im.add_networkx_graph(G)
im.run()

# Assign detected modules back to G
nx.set_node_attributes(G, im.get_modules(), "node_color_attr")

# ---------------------------------------------------------------------
# 2. Draw the network
# ---------------------------------------------------------------------

util.plot_modular_network(
    G,
    pos=pos,
    node_color_attr="node_color_attr",
    figsize=(6, 4),
    edge_width=1.5,
    use_value_as_palette_index=True,
    shape_attr="metadata",
    shape_map={1: "s", 2: "o"},
)

## Modules from a metadata-dependent encoding of random walks

Now let the metadata shape the flow. For a walker at node $u$ stepping to node $v$, you encode the step with probability $\epsilon(\ell_u, \ell_v)$, which depends on the labels at both ends; otherwise the walker keeps moving without encoding until a step is finally encoded. Summed over many walks, this yields a metadata-dependent encoding graph: the same nodes, but link weights set by how often each step gets encoded. Running Infomap on that graph can group nodes by label even when they sit in different structural modules.

The Rust simulator builds the encoding graph for you. The two probability arguments shape $\epsilon$. The first is the baseline weight $p$; setting `same_prob` $> 1$ favors assortative modules, putting same-label nodes together, while $p <$ `same_prob` $< 1$ favors disassortative modules. The cell writes the directed edge list and the labels, runs the simulation to produce a metadata-weighted flow network, then detects modules with Infomap using the `rawdir` flow model.


In [ ]:
# ---------------------------------------------------------------------
# 1. Prepare data for the metadata-aware simulation
# ---------------------------------------------------------------------

# Baseline encoding probability p ∈ [0, 1]
p = 0.09

# Infomap requires directed edges for rawdir flow model
# Convert the undirected graph G into a directed version H
H = nx.DiGraph()
for u, v in G.edges:
    H.add_edge(u, v)
    H.add_edge(v, u)

# Write the directed edge list to file
nx.write_edgelist(H, "output/fig0.net", data=False)

# Write node metadata to a separate file.
# Format: "<node> <metadata_value>"
with open("output/fig0-meta.txt", "w") as fp:
    for node, meta in G.nodes.data("metadata"):
        fp.write(f"{node} {meta}\n")

# ---------------------------------------------------------------------
# 2. Run the metadata-dependent random walk simulation
# ---------------------------------------------------------------------
# The Rust simulator takes:
#   - metadata type flag:   "-c" (categorical) or "-r" (real-valued)
#   - input edge list:      fig0.net
#   - metadata input file:  fig0-meta.txt
#   - output path:          fig0-meta.net
#   - same_prob:            a) 1.0 (neutral encoding)
#                           b) >1 (assortative encoding)
#                           c) >p and <1 (disassortative encoding)
#   - diff_prob:            if <<1, the actual structure of the graph becomes less and less relevant
#                           if ~1, categorical information becomes irrelevant
#   - n_samples:            number of random walk steps

!cargo run --manifest-path color-map-equation/simulate/Cargo.toml -- \
    -c output/fig0.net output/fig0-meta.txt output/fig0-meta.net 1.0 {p} 1000000


# ---------------------------------------------------------------------
# 3. Detect communities using Infomap with simulated metadata-aware flow
# ---------------------------------------------------------------------

im = Infomap(flow_model="rawdir", silent=True, num_trials=10, seed=123)

# Load the metadata-weighted flow network produced by the simulation
im.read_file("output/fig0-meta.net")
im.run()

# Assign detected module labels back to the original undirected graph G
nx.set_node_attributes(G, im.get_modules(), "node_color_attr")

# ---------------------------------------------------------------------
# 4. Plot the network colored by community and shaped by metadata
# ---------------------------------------------------------------------

util.plot_modular_network(
    G,
    pos=pos,
    node_color_attr="node_color_attr",
    figsize=(6, 4),
    edge_width=1.5,
    use_value_as_palette_index=True,
    shape_attr="metadata",
    shape_map={1: "s", 2: "o"},
)